# ranking_forcast — Predizione per la prossima stagione

La finestra storica viene letta da `configs/config.yaml`: da `first_year` a `last_year`, estremi inclusi. La previsione riguarda la stagione che inizia in `last_year + 1` e comprende tutti i coach del dataset.

Punteggi: **W = 20**, **Q = 6** (secondo o terzo posto, escluso il vincitore), **N = 1**, **A (assente) = 1**, **R = -2**. Questi pesi sono specifici della previsione e indipendenti dalla conversione usata nei grafici dello storico.

- **A_storico**: somma dei punteggi delle sole stagioni giocate (`W`, `Q`, `N`, `R`), divisa per il numero di partecipazioni. Le assenze non entrano né nella somma né nel denominatore storico.
- **B_ultimi_3**: somma dei punteggi delle ultime tre stagioni della finestra, divisa per 3. Assenze e stagioni precedenti all'inizio della finestra valgono 1.
- **Nessuna partecipazione** nella finestra (anche quando tutte le stagioni sono assenze): entrambi i valori sono `(R + N + Q) / 3 = 5/3`.
- **ranking_forcast**: `(A_storico + B_ultimi_3) / T * 100`, dove T è la somma massima. Le parità usano posizioni come `1, 2, 2, 4`, calcolate prima dell'arrotondamento visuale.

Se T è zero, la normalizzazione non è definita e viene segnalato un errore. Con T negativo la formula inverte l'ordine dei valori normalizzati: la classifica resta ordinata per somma A + B. I punteggi negativi non vengono tagliati.


In [1]:
from pathlib import Path
from fractions import Fraction
import sys

import pandas as pd
import yaml

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_results


## Configurazione e dati

In [2]:
with (PROJECT_ROOT / "configs" / "config.yaml").open(encoding="utf-8") as stream:
    config = yaml.safe_load(stream)

years = config["year_selection"]
first_year = years["first_year"]
last_year = years["last_year"]
results = load_results(
    PROJECT_ROOT / config["data"]["file_path"], first_year, last_year
)

forecast_year = last_year + 1
print(f"Storico: {first_year}/{first_year + 1} – {last_year}/{last_year + 1}")
print(f"Predizione: {forecast_year}/{forecast_year + 1}")
print(f"Coach: {len(results.columns)}")


Storico: 2020/2021 – 2025/2026
Predizione: 2026/2027
Coach: 14


## Calcolo della classifica

In [3]:
FORECAST_POINTS = {"W": 20, "Q": 6, "N": 1, "A": 1, "R": -2}


def build_ranking_forcast(results: pd.DataFrame, last_year: int) -> pd.DataFrame:
    """Calcola la previsione sui risultati già selezionati tramite config."""
    recent_years = range(last_year - 2, last_year + 1)
    newcomer_score = Fraction(sum(FORECAST_POINTS[r] for r in ("R", "N", "Q")), 3)
    rows = []
    for coach in results.columns:
        history = results[coach]
        played = history[history != "A"]
        participations = len(played)
        if participations == 0:
            historical = recent = newcomer_score
        else:
            historical = Fraction(sum(FORECAST_POINTS[r] for r in played), participations)
            recent_results = history.reindex(recent_years, fill_value="A")
            recent = Fraction(sum(FORECAST_POINTS[r] for r in recent_results), 3)
        rows.append({
            "Nome": coach,
            "Partecipazioni": participations,
            "A_storico": historical,
            "B_ultimi_3": recent,
            "Somma": historical + recent,
        })

    columns = ["Posizione", "Nome", "Partecipazioni", "A_storico", "B_ultimi_3", "Somma", "ranking_forcast"]
    if not rows:
        return pd.DataFrame(columns=columns)

    # Frazioni esatte: somme matematicamente uguali ricevono la stessa posizione.
    rows.sort(key=lambda row: row["Somma"], reverse=True)
    top_score = rows[0]["Somma"]
    if top_score == 0:
        raise ValueError("Impossibile normalizzare: la somma massima T è zero.")

    previous_score = None
    position = 0
    for index, row in enumerate(rows, start=1):
        total = row["Somma"]
        if total != previous_score:
            position = index
        row["Posizione"] = position
        row["ranking_forcast"] = float(total / top_score * 100)
        previous_score = total
        for column in ("A_storico", "B_ultimi_3", "Somma"):
            row[column] = float(row[column])
    return pd.DataFrame(rows, columns=columns)


In [4]:
ranking_forcast = build_ranking_forcast(results, last_year)
print(f"ranking_forcast — stagione {forecast_year}/{forecast_year + 1}")
display(ranking_forcast.style.hide(axis="index").format({
    "A_storico": "{:.3f}",
    "B_ultimi_3": "{:.3f}",
    "Somma": "{:.3f}",
    "ranking_forcast": "{:.2f}",
}))


ranking_forcast — stagione 2026/2027


Posizione,Nome,Partecipazioni,A_storico,B_ultimi_3,Somma,ranking_forcast
1,Gallo,6,4.333,8.000,12.333,100.00
2,Zanu,6,3.667,7.333,11.000,89.19
3,Pippi,6,3.500,5.333,8.833,71.62
4,Javier,1,6.000,2.667,8.667,70.27
5,Lippo,6,5.333,2.667,8.000,64.86
6,Manu,6,5.333,1.667,7.000,56.76
7,Cucu,6,3.500,2.667,6.167,50.00
8,Cappo,6,4.000,0.667,4.667,37.84
9,Furlan,3,1.000,1.000,2.000,16.22
9,Alex,1,1.000,1.000,2.000,16.22


## Verifica delle regole
Esegui questi esempi per controllare i casi particolari.

In [5]:
# Esempi sintetici: assenze, storico, finestra breve, esordienti e parità.
from math import isclose

sample = pd.DataFrame({
    "Vincitore": ["W", "W", "W", "W"],
    "Pippo": ["W", "A", "N", "Q"],
    "Pluto": ["W", "A", "N", "Q"],
    "Topolino": ["A", "A", "A", "A"],
}, index=[2022, 2023, 2024, 2025])
check = build_ranking_forcast(sample, 2025).set_index("Nome")
assert check["Posizione"].tolist() == [1, 2, 2, 4]
assert check.loc["Vincitore", "ranking_forcast"] == 100
assert check.loc["Pippo", "A_storico"] == 9
assert isclose(check.loc["Pippo", "B_ultimi_3"], 8 / 3)
assert isclose(check.loc["Topolino", "A_storico"], 5 / 3)
assert isclose(check.loc["Topolino", "B_ultimi_3"], 5 / 3)

for values, expected in [(["W"], 22 / 3), (["W", "Q"], 27 / 3)]:
    short = pd.DataFrame({"Coach": values}, index=range(2026 - len(values), 2026))
    assert isclose(build_ranking_forcast(short, 2025).iloc[0]["B_ultimi_3"], expected)

empty_history = pd.DataFrame(columns=["Esordiente"], index=pd.Index([], dtype=int))
newcomer = build_ranking_forcast(empty_history, 2025).iloc[0]
assert isclose(newcomer["A_storico"], 5 / 3)
assert isclose(newcomer["B_ultimi_3"], 5 / 3)

relegated = pd.DataFrame({"Coach": ["R"]}, index=[2025])
assert build_ranking_forcast(relegated, 2025).iloc[0]["A_storico"] == -2
zero = pd.DataFrame({"Coach": ["R", "N", "N"]}, index=[2023, 2024, 2025])
try:
    build_ranking_forcast(zero, 2025)
except ValueError as error:
    assert "T è zero" in str(error)
else:
    raise AssertionError("T = 0 deve segnalare normalizzazione non definita")
print("Verifiche superate.")


Verifiche superate.
